# Compare road-extraction models - Colab

Enter one lat/lon, download one tile, run `baseline` / `dlinknet` / `unet` on it, and see
all three results side by side (overlay + mask, each panel titled with the model's real
name). `dlinknet` and `unet` are pretrained neural nets; `baseline` is a classical filter
with no weights - it's the free sanity floor, not a model to trust.

Run all cells top to bottom. Cell 6 will ask you for coordinates.

`Runtime` -> `Change runtime type` -> `T4 GPU` speeds up dlinknet/unet, but isn't required.


In [ ]:
# Colab already has numpy, scipy, pillow, requests, torch, torchvision.
!pip -q install rasterio shapely pyproj scikit-image

import os
os.chdir("/content")
os.makedirs("CNN", exist_ok=True)
os.chdir("CNN")

# models/weights/ is not tracked by git in the source project, so a fresh clone/rebuild
# never has it - create every directory explicitly instead of assuming it exists.
for d in ["core", "models/weights", "data/images", "data/osm",
          "outputs/images", "outputs/geojson", "reports"]:
    os.makedirs(d, exist_ok=True)
print("directories ready")


In [ ]:
# The exact project files, tested locally - written verbatim, not rewritten.
FILES = {
    'config.py': '"""\nAll fixed settings for the project.\nChange values HERE and nowhere else.\n"""\nfrom pathlib import Path\n\nROOT = Path(__file__).parent\n\n# ---------- folders ----------\nDATA_IMAGES = ROOT / "data" / "images"     # downloaded satellite GeoTIFFs\nOSM_DIR     = ROOT / "data" / "osm"        # put bangladesh-260823.osm.pbf here\nOUT_IMAGES  = ROOT / "outputs" / "images"  # overlay pictures  <name>_<model>.png\nOUT_GEOJSON = ROOT / "outputs" / "geojson" # road vectors      <name>_<model>.geojson\nREPORTS     = ROOT / "reports"             # validation.csv\nWEIGHTS     = ROOT / "models" / "weights"  # put .pth files here\n\n# ---------- imagery standard (hardcoded on purpose) ----------\nZOOM     = 18     # ~0.55 m/pixel in Bangladesh -> matches DeepGlobe/SpaceNet training data\nSIZE_PX  = 2048   # downloaded image is 2048x2048  (~1.1 km x 1.1 km)\nPATCH_PX = 512    # models see 512x512 patches (16 patches per image)\nOVERLAP_PX = 96   # patches overlap by this much; blended so seams disappear\n\nBASEMAP = ("https://services.arcgisonline.com/ArcGIS/rest/services/"\n           "World_Imagery/MapServer/tile/{z}/{y}/{x}")\nUSER_AGENT = "road-extraction-research/0.1"\n\n# ---------- mask -> vector ----------\nTHRESHOLD   = 0.5   # probability above this = road\nMIN_ROAD_PX = 40    # delete blobs smaller than this (removes specks)\nSIMPLIFY_M  = 2.0   # line smoothing tolerance, metres\n\n# ---------- validation ----------\nOSM_PBF        = OSM_DIR / "bangladesh-260823.osm.pbf"\nOSM_ROAD_WIDTH = 4.0   # metres, half-width used to draw OSM roads as a mask\nMATCH_SLACK_PX = 8     # a prediction within this many pixels of OSM counts as a match\n\nfor _d in (DATA_IMAGES, OSM_DIR, OUT_IMAGES, OUT_GEOJSON, REPORTS, WEIGHTS):\n    _d.mkdir(parents=True, exist_ok=True)\n',
    'core/__init__.py': '',
    'core/geo.py': '"""\nCoordinate maths.\n\nTwo coordinate systems are used:\n  EPSG:4326 - plain lat/lon degrees   (what iD / JOSM / GeoJSON use)\n  EPSG:3857 - Web Mercator metres     (what map tiles use)\n"""\nimport math\n\nTILE_PX = 256\nORIGIN = 20037508.342789244   # half the width of the Web Mercator world, in metres\n\n\ndef latlon_to_pixel(lat, lon, zoom):\n    """Global pixel coordinate of a lat/lon at a given zoom level."""\n    n = TILE_PX * (2 ** zoom)\n    x = (lon + 180.0) / 360.0 * n\n    lat_r = math.radians(lat)\n    y = (1.0 - math.log(math.tan(lat_r) + 1.0 / math.cos(lat_r)) / math.pi) / 2.0 * n\n    return x, y\n\n\ndef pixel_to_latlon(x, y, zoom):\n    """Inverse of latlon_to_pixel."""\n    n = TILE_PX * (2 ** zoom)\n    lon = x / n * 360.0 - 180.0\n    lat = math.degrees(math.atan(math.sinh(math.pi * (1.0 - 2.0 * y / n))))\n    return lat, lon\n\n\ndef mercator_res(zoom):\n    """Metres per pixel in EPSG:3857 (constant, not corrected for latitude)."""\n    return 2.0 * ORIGIN / (TILE_PX * (2 ** zoom))\n\n\ndef ground_res(lat, zoom):\n    """Real metres per pixel on the ground at this latitude. This is the GSD."""\n    return 156543.03392 * math.cos(math.radians(lat)) / (2 ** zoom)\n\n\ndef raster_center_latlon(src):\n    """Centre of an open rasterio dataset, as (lat, lon)."""\n    from pyproj import Transformer\n    b = src.bounds\n    cx, cy = (b.left + b.right) / 2.0, (b.bottom + b.top) / 2.0\n    t = Transformer.from_crs(src.crs, "EPSG:4326", always_xy=True)\n    lon, lat = t.transform(cx, cy)\n    return lat, lon\n\n\ndef raster_bounds_latlon(src):\n    """Bounding box of an open rasterio dataset as (west, south, east, north) in degrees."""\n    from pyproj import Transformer\n    b = src.bounds\n    t = Transformer.from_crs(src.crs, "EPSG:4326", always_xy=True)\n    west, south = t.transform(b.left, b.bottom)\n    east, north = t.transform(b.right, b.top)\n    return west, south, east, north\n',
    'core/tiling.py': '"""\nRun a patch-based model over a big image without seams at patch edges.\n\nA model only sees one 512x512 patch at a time, so a road running along a\npatch boundary gets cut with no context on one side - the prediction is\nweakest exactly at every seam. The fix costs no training: overlap the\npatches and blend the overlapping region, weighted so each pixel is\ninfluenced most by the patch where it sits closest to the centre (a Hann\nwindow). Free accuracy, pretrained model unchanged.\n"""\nimport numpy as np\n\n\ndef _hann_window(size):\n    w1d = np.hanning(size + 2)[1:-1]           # avoid the zero endpoints\n    return np.outer(w1d, w1d).astype(np.float32)\n\n\ndef tiled_predict(model, image, patch_px, overlap_px):\n    """Run model.predict() over overlapping patches and blend the results."""\n    h, w = image.shape[:2]\n    stride = patch_px - overlap_px\n    window = _hann_window(patch_px)\n\n    acc = np.zeros((h, w), np.float32)\n    weight = np.zeros((h, w), np.float32)\n\n    rows = list(range(0, max(h - patch_px, 0) + 1, stride)) or [0]\n    cols = list(range(0, max(w - patch_px, 0) + 1, stride)) or [0]\n    if rows[-1] + patch_px < h:\n        rows.append(h - patch_px)\n    if cols[-1] + patch_px < w:\n        cols.append(w - patch_px)\n\n    for r in rows:\n        for c in cols:\n            tile = image[r:r + patch_px, c:c + patch_px]\n            th, tw = tile.shape[:2]\n            if (th, tw) != (patch_px, patch_px):\n                pad = np.zeros((patch_px, patch_px, 3), tile.dtype)\n                pad[:th, :tw] = tile\n                tile = pad\n\n            out = model.predict(tile)\n            acc[r:r + th, c:c + tw] += (out * window)[:th, :tw]\n            weight[r:r + th, c:c + tw] += window[:th, :tw]\n\n    weight[weight == 0] = 1.0\n    return (acc / weight).astype(np.float32)\n',
    'core/vectorize.py': '"""\nTurn a road probability map into GeoJSON lines.\n\n    probability (float 0-1)\n      -> threshold        -> binary mask\n      -> remove specks\n      -> skeletonize      -> 1-pixel-wide centre lines\n      -> trace            -> ordered pixel paths\n      -> simplify         -> fewer points\n      -> reproject        -> lat/lon LineStrings\n"""\nimport inspect\nimport json\nimport math\n\nimport numpy as np\nfrom shapely.geometry import LineString\nfrom skimage.draw import line as _draw_line\nfrom skimage.morphology import remove_small_objects, skeletonize\n\n_NB = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]\n\n\n_RSO_ARGS = inspect.signature(remove_small_objects).parameters\n\n\ndef prob_to_mask(prob, threshold, min_px):\n    """Threshold the probability map and delete blobs smaller than min_px."""\n    mask = prob >= threshold\n    if min_px > 0:\n        # scikit-image renamed min_size -> max_size in 0.26 (and changed it to\n        # "smaller than or equal to"), so support both spellings\n        if "max_size" in _RSO_ARGS:\n            mask = remove_small_objects(mask, max_size=min_px - 1)\n        else:\n            mask = remove_small_objects(mask, min_size=min_px)\n    return mask\n\n\ndef _neighbors(sk, r, c):\n    h, w = sk.shape\n    out = []\n    for dr, dc in _NB:\n        rr, cc = r + dr, c + dc\n        if 0 <= rr < h and 0 <= cc < w and sk[rr, cc]:\n            out.append((rr, cc))\n    return out\n\n\ndef mask_to_paths(mask):\n    """Skeletonize the mask and return a list of pixel paths [(row, col), ...]."""\n    sk = skeletonize(mask.astype(bool))\n    pts = list(zip(*np.nonzero(sk)))\n    if not pts:\n        return []\n\n    degree = {p: len(_neighbors(sk, *p)) for p in pts}\n    nodes = [p for p in pts if degree[p] != 2]          # endpoints and junctions\n\n    paths, started = [], set()\n\n    def walk(a, b):\n        path, prev, cur = [a, b], a, b\n        while degree.get(cur, 0) == 2:\n            nxt = [n for n in _neighbors(sk, *cur) if n != prev]\n            if not nxt:\n                break\n            prev, cur = cur, nxt[0]\n            path.append(cur)\n        return path\n\n    for n in nodes:\n        for nb in _neighbors(sk, *n):\n            if frozenset((n, nb)) in started:\n                continue\n            p = walk(n, nb)\n            started.add(frozenset((p[0], p[1])))\n            started.add(frozenset((p[-1], p[-2])))\n            paths.append(p)\n\n    # closed loops have no endpoint or junction, so the walk above never reaches them\n    left = set(pts) - {q for p in paths for q in p}\n    while left:\n        start = next(iter(left))\n        left.discard(start)\n        path, cur = [start], start\n        while True:\n            nxt = [n for n in _neighbors(sk, *cur) if n in left]\n            if not nxt:\n                break\n            cur = nxt[0]\n            left.discard(cur)\n            path.append(cur)\n        if len(path) > 1:\n            paths.append(path)\n\n    return paths\n\n\ndef paths_to_mask(paths, shape):\n    """Rasterize pixel paths back into a mask. Only used for the overlay picture."""\n    mask = np.zeros(shape, bool)\n    h, w = shape\n    for path in paths:\n        for (r0, c0), (r1, c1) in zip(path, path[1:]):\n            rr, cc = _draw_line(int(r0), int(c0), int(r1), int(c1))\n            keep = (rr >= 0) & (rr < h) & (cc >= 0) & (cc < w)\n            mask[rr[keep], cc[keep]] = True\n    return mask\n\n\ndef paths_to_geojson(paths, transform, crs, center_lat,\n                     simplify_m=2.0, min_len_m=15.0, props=None):\n    """Convert pixel paths to a GeoJSON FeatureCollection in EPSG:4326."""\n    from pyproj import Transformer\n\n    to_wgs = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)\n    # Web Mercator exaggerates distance by 1/cos(lat); undo it for real metres\n    scale = math.cos(math.radians(center_lat))\n\n    features = []\n    for path in paths:\n        if len(path) < 2:\n            continue\n        xy = [transform * (c + 0.5, r + 0.5) for r, c in path]\n        line = LineString(xy).simplify(simplify_m)\n        length_m = line.length * scale\n        if length_m < min_len_m:\n            continue\n\n        xs = [p[0] for p in line.coords]\n        ys = [p[1] for p in line.coords]\n        lons, lats = to_wgs.transform(xs, ys)\n        coords = [[round(float(a), 7), round(float(b), 7)] for a, b in zip(lons, lats)]\n\n        p = dict(props or {})\n        p["length_m"] = round(float(length_m), 1)\n        features.append({"type": "Feature", "properties": p,\n                         "geometry": {"type": "LineString", "coordinates": coords}})\n\n    return {"type": "FeatureCollection", "features": features}\n\n\ndef save_geojson(fc, path):\n    with open(path, "w") as f:\n        json.dump(fc, f)\n    return path\n',
    'core/viz.py': '"""Overlay pictures so you can judge the result by eye."""\nimport numpy as np\nfrom PIL import Image\n\n\ndef save_overlay(image_rgb, mask, path, color=(255, 40, 40), alpha=0.55):\n    """Paint the predicted mask on top of the satellite image and save a PNG."""\n    out = image_rgb.astype(np.float32).copy()\n    tint = np.array(color, dtype=np.float32)\n    sel = mask.astype(bool)\n    out[sel] = (1.0 - alpha) * out[sel] + alpha * tint\n    Image.fromarray(out.clip(0, 255).astype(np.uint8)).save(path)\n    return path\n\n\ndef save_mask(mask, path):\n    Image.fromarray((mask.astype(np.uint8) * 255)).save(path)\n    return path\n',
    'core/registry.py': '"""\nFinds every model automatically.\n\nDrop a new file in models/ that defines a class inheriting RoadModel,\nand it becomes available immediately. No other file needs editing.\nFiles starting with \'_\' are ignored.\n"""\nimport importlib\nimport pkgutil\nfrom pathlib import Path\n\nimport models\nfrom models._base import RoadModel\n\n\ndef discover():\n    """Return {model_name: model_class} for every model file in models/."""\n    found = {}\n    pkg_dir = Path(models.__file__).parent\n    for info in pkgutil.iter_modules([str(pkg_dir)]):\n        if info.name.startswith("_"):\n            continue\n        module = importlib.import_module(f"models.{info.name}")\n        for obj in vars(module).values():\n            if (isinstance(obj, type) and issubclass(obj, RoadModel)\n                    and obj is not RoadModel):\n                found[obj.name] = obj\n    return found\n',
    'models/__init__.py': '',
    'models/_base.py': '"""\nThe one interface every model must follow.\n\nTo add a model, create models/<yourmodel>.py containing:\n\n    from models._base import RoadModel\n\n    class YourModel(RoadModel):\n        name = "yourmodel"          # used in output filenames\n\n        def load(self):\n            ...                     # load weights once\n\n        def predict(self, patch):\n            ...                     # return float32 HxW, values 0..1\n\nNothing else in the project needs to change.\n"""\nimport numpy as np\n\n\nclass RoadModel:\n    name = "unnamed"\n    description = ""\n\n    # "mask"  -> implement predict();       the pipeline skeletonizes for you\n    # "graph" -> implement predict_graph(); the model gives lines directly\n    outputs = "mask"\n\n    def load(self):\n        """Called once before the first predict(). Load weights here."""\n        return self\n\n    def predict(self, patch: np.ndarray) -> np.ndarray:\n        """\n        Mask models implement this. Called once per 512x512 patch.\n\n        patch  : uint8 array, shape (H, W, 3), RGB\n        returns: float32 array, shape (H, W), 0.0 = not road, 1.0 = road\n        """\n        raise NotImplementedError\n\n    def predict_graph(self, image: np.ndarray):\n        """\n        Graph models implement this instead. Called once on the whole image.\n\n        image  : uint8 array, shape (H, W, 3), RGB\n        returns: list of paths, each a list of (row, col) pixel pairs\n        """\n        raise NotImplementedError\n',
    'models/baseline.py': '"""\nClassical baseline. No weights, no GPU, no download - it runs today.\n\nUses a ridge filter (Frangi), originally built to find blood vessels in\nmedical scans. Roads are also long thin bright ridges, so it half-works.\n\nPurpose: prove the whole pipeline end to end before you fight with PyTorch.\nEvery deep model you add should beat this. If one does not, something is wrong.\n"""\nimport numpy as np\nfrom skimage.color import rgb2gray\nfrom skimage.filters import frangi\n\nfrom models._base import RoadModel\n\n\nclass BaselineRidge(RoadModel):\n    name = "baseline"\n    description = "Frangi ridge filter, no training required"\n\n    def predict(self, patch):\n        gray = rgb2gray(patch)\n        # sigmas roughly match road half-widths in pixels at 0.55 m/px\n        resp = frangi(gray, sigmas=range(2, 12, 2), black_ridges=False)\n        # Normalise by a high percentile, not by the maximum. One bright\n        # artefact would otherwise push every real road down towards zero.\n        hi = np.percentile(resp, 99.0)\n        if hi > 0:\n            resp = np.clip(resp / hi, 0.0, 1.0)\n        return resp.astype(np.float32)\n',
    'models/dlinknet.py': '"""\nD-LinkNet34 - winner of the DeepGlobe 2018 Road Extraction Challenge.\n\nResNet34 encoder + a dilated centre block + LinkNet decoder. The dilated\nblock is the whole idea: it widens the receptive field without losing\nresolution, so the network can follow a road that disappears under trees\nand reappears further along.\n\nPaper:   https://openaccess.thecvf.com/content_cvpr_2018_workshops/papers/w4/Zhou_D-LinkNet_LinkNet_With_CVPR_2018_paper.pdf\nWeights: models/weights/dlinknet34_deepglobe.th  (trained on DeepGlobe, 0.5 m/px)\n\nTrained on rural/suburban roads in Thailand, India and Indonesia, so it\ntransfers to Bangladesh better than any city-trained model.\n"""\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torchvision.models import resnet34\n\nimport config\nfrom models._base import RoadModel\n\nWEIGHTS = config.WEIGHTS / "dlinknet34_deepglobe.th"\n\n\nclass Dblock(nn.Module):\n    """Dilated centre block: four 3x3 convs with dilation 1, 2, 4, 8, summed."""\n\n    def __init__(self, ch):\n        super().__init__()\n        self.dilate1 = nn.Conv2d(ch, ch, 3, dilation=1, padding=1)\n        self.dilate2 = nn.Conv2d(ch, ch, 3, dilation=2, padding=2)\n        self.dilate3 = nn.Conv2d(ch, ch, 3, dilation=4, padding=4)\n        self.dilate4 = nn.Conv2d(ch, ch, 3, dilation=8, padding=8)\n\n    def forward(self, x):\n        d1 = F.relu(self.dilate1(x))\n        d2 = F.relu(self.dilate2(d1))\n        d3 = F.relu(self.dilate3(d2))\n        d4 = F.relu(self.dilate4(d3))\n        return x + d1 + d2 + d3 + d4\n\n\nclass DecoderBlock(nn.Module):\n    """1x1 squeeze -> transposed conv (x2) -> 1x1 expand."""\n\n    def __init__(self, in_ch, out_ch):\n        super().__init__()\n        mid = in_ch // 4\n        self.conv1 = nn.Conv2d(in_ch, mid, 1)\n        self.norm1 = nn.BatchNorm2d(mid)\n        self.deconv2 = nn.ConvTranspose2d(mid, mid, 3, stride=2,\n                                          padding=1, output_padding=1)\n        self.norm2 = nn.BatchNorm2d(mid)\n        self.conv3 = nn.Conv2d(mid, out_ch, 1)\n        self.norm3 = nn.BatchNorm2d(out_ch)\n\n    def forward(self, x):\n        x = F.relu(self.norm1(self.conv1(x)))\n        x = F.relu(self.norm2(self.deconv2(x)))\n        return F.relu(self.norm3(self.conv3(x)))\n\n\nclass DinkNet34(nn.Module):\n    def __init__(self, num_classes=1):\n        super().__init__()\n        res = resnet34(weights=None)\n        self.firstconv = res.conv1\n        self.firstbn = res.bn1\n        self.firstrelu = res.relu\n        self.firstmaxpool = res.maxpool\n        self.encoder1, self.encoder2 = res.layer1, res.layer2\n        self.encoder3, self.encoder4 = res.layer3, res.layer4\n\n        self.dblock = Dblock(512)\n\n        self.decoder4 = DecoderBlock(512, 256)\n        self.decoder3 = DecoderBlock(256, 128)\n        self.decoder2 = DecoderBlock(128, 64)\n        self.decoder1 = DecoderBlock(64, 64)\n\n        self.finaldeconv1 = nn.ConvTranspose2d(64, 32, 4, 2, 1)\n        self.finalconv2 = nn.Conv2d(32, 32, 3, padding=1)\n        self.finalconv3 = nn.Conv2d(32, num_classes, 3, padding=1)\n\n    def forward(self, x):\n        x = self.firstmaxpool(self.firstrelu(self.firstbn(self.firstconv(x))))\n        e1 = self.encoder1(x)\n        e2 = self.encoder2(e1)\n        e3 = self.encoder3(e2)\n        e4 = self.dblock(self.encoder4(e3))\n\n        d4 = self.decoder4(e4) + e3\n        d3 = self.decoder3(d4) + e2\n        d2 = self.decoder2(d3) + e1\n        d1 = self.decoder1(d2)\n\n        out = F.relu(self.finaldeconv1(d1))\n        out = F.relu(self.finalconv2(out))\n        return torch.sigmoid(self.finalconv3(out))\n\n\nclass DLinkNet34(RoadModel):\n    name = "dlinknet"\n    description = "D-LinkNet34, DeepGlobe winner (rural roads)"\n\n    tta = True    # 4-way flip averaging: meaningfully better, ~4x slower - fine on CPU\n\n    def load(self):\n        if not WEIGHTS.exists():\n            raise FileNotFoundError(\n                f"Missing weights: {WEIGHTS}\\n"\n                "Download log01_dink34.th from the D-LinkNet repo and save it there."\n            )\n        self.device = "cuda" if torch.cuda.is_available() else "cpu"\n        state = torch.load(WEIGHTS, map_location="cpu", weights_only=False)\n        # the checkpoint was saved from nn.DataParallel, so keys start with "module."\n        state = {k.replace("module.", "", 1): v for k, v in state.items()}\n\n        self.net = DinkNet34()\n        self.net.load_state_dict(state)\n        self.net.to(self.device).eval()\n        torch.set_num_threads(max(1, torch.get_num_threads()))\n        return self\n\n    def _prep(self, patch):\n        # the original repo read images with cv2 (BGR) and scaled to [-1.6, 1.6]\n        bgr = patch[:, :, ::-1].astype(np.float32)\n        return bgr / 255.0 * 3.2 - 1.6\n\n    def predict(self, patch):\n        x = self._prep(patch)\n        batch = [x]\n        if self.tta:\n            batch = [x, x[::-1], x[:, ::-1], x[::-1, ::-1]]\n\n        arr = np.ascontiguousarray(np.stack(batch).transpose(0, 3, 1, 2))\n        with torch.no_grad():\n            out = self.net(torch.from_numpy(arr).to(self.device))\n        out = out[:, 0].cpu().numpy()\n\n        if self.tta:\n            out = np.stack([out[0], out[1][::-1], out[2][:, ::-1],\n                            out[3][::-1, ::-1]]).mean(0)\n        else:\n            out = out[0]\n        return out.astype(np.float32)\n',
    'models/unet.py': '"""\nPlain U-Net (Ronneberger et al.) - 4-level encoder/decoder, skip connections,\nno pretrained backbone. Trained on the Massachusetts Roads Dataset.\n\nWeights: models/weights/unet_road_massachusetts.pth\nSource:  https://huggingface.co/teohyc/Satellite-Road-Segmentation-UNet\n         (checkpoint verified tensor-by-tensor against this exact class - 136\n         tensors, all keys and shapes match)\n\nIMPORTANT - resolution mismatch, read before trusting the results\n-------------------------------------------------------------------\nThe training pipeline resizes each full 1500x1500 px tile (1 m/px Massachusetts\naerial imagery) down to 256x256 before feeding the network - i.e. every image\nthe network has ever seen is effectively ~5.9 m/px, no matter how sharp the\nsource photo was. That is far coarser than our 0.55 m/px Bangladesh tiles.\n\nTo match that scale as closely as this pipeline\'s patch-based design allows,\neach 512x512 patch (0.55 m/px, ~280 m across) is resized down to 256x256\nbefore inference - about 1.1 m/px, still roughly 5x sharper than what the\nnetwork was trained on. In practice this means it may see our roads as\nunusually thin and under-detect them; unlike dlinknet, this is not a strong\nmodel to rely on for Bangladesh imagery. See README.md.\n"""\nimport numpy as np\nimport torch\nimport torch.nn as nn\n\nimport config\nfrom models._base import RoadModel\n\nWEIGHTS = config.WEIGHTS / "unet_road_massachusetts.pth"\nNET_SIZE = 256   # fixed by how the checkpoint was trained\n\n\nclass ConvBlock(nn.Module):\n    def __init__(self, in_channels, out_channels):\n        super().__init__()\n        self.conv = nn.Sequential(\n            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),\n            nn.BatchNorm2d(out_channels),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),\n            nn.BatchNorm2d(out_channels),\n            nn.ReLU(inplace=True),\n            nn.Dropout(0.3),\n        )\n\n    def forward(self, x):\n        return self.conv(x)\n\n\nclass UNet(nn.Module):\n    def __init__(self, in_channels=3, out_channels=1):\n        super().__init__()\n        self.enc1 = ConvBlock(in_channels, 64)\n        self.enc2 = ConvBlock(64, 128)\n        self.enc3 = ConvBlock(128, 256)\n        self.enc4 = ConvBlock(256, 512)\n        self.pool = nn.MaxPool2d(2)\n\n        self.bottleneck = ConvBlock(512, 1024)\n\n        self.upconv4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)\n        self.dec4 = ConvBlock(1024, 512)\n        self.upconv3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)\n        self.dec3 = ConvBlock(512, 256)\n        self.upconv2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)\n        self.dec2 = ConvBlock(256, 128)\n        self.upconv1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)\n        self.dec1 = ConvBlock(128, 64)\n\n        self.conv_final = nn.Conv2d(64, out_channels, kernel_size=1)\n\n    def forward(self, x):\n        e1 = self.enc1(x)\n        e2 = self.enc2(self.pool(e1))\n        e3 = self.enc3(self.pool(e2))\n        e4 = self.enc4(self.pool(e3))\n\n        b = self.bottleneck(self.pool(e4))\n\n        d4 = self.dec4(torch.cat([self.upconv4(b), e4], dim=1))\n        d3 = self.dec3(torch.cat([self.upconv3(d4), e3], dim=1))\n        d2 = self.dec2(torch.cat([self.upconv2(d3), e2], dim=1))\n        d1 = self.dec1(torch.cat([self.upconv1(d2), e1], dim=1))\n\n        return torch.sigmoid(self.conv_final(d1))\n\n\nclass UNetRoad(RoadModel):\n    name = "unet"\n    description = "Plain U-Net, Massachusetts Roads Dataset (scale mismatch - see file docstring)"\n\n    def load(self):\n        if not WEIGHTS.exists():\n            raise FileNotFoundError(\n                f"Missing weights: {WEIGHTS}\\n"\n                "Download best_road_seg_unet.pth from "\n                "https://huggingface.co/teohyc/Satellite-Road-Segmentation-UNet"\n            )\n        self.device = "cuda" if torch.cuda.is_available() else "cpu"\n        state = torch.load(WEIGHTS, map_location="cpu", weights_only=False)\n\n        self.net = UNet(in_channels=3, out_channels=1)\n        self.net.load_state_dict(state)\n        self.net.to(self.device).eval()\n        return self\n\n    def predict(self, patch):\n        ph, pw = patch.shape[:2]\n        x = torch.from_numpy(patch.astype(np.float32) / 255.0)\n        x = x.permute(2, 0, 1).unsqueeze(0)                          # 1x3xHxW\n        x = torch.nn.functional.interpolate(x, size=(NET_SIZE, NET_SIZE),\n                                            mode="bilinear", align_corners=False)\n        x = x.to(self.device)\n\n        with torch.no_grad():\n            out = self.net(x)\n            out = torch.nn.functional.interpolate(out, size=(ph, pw),\n                                                  mode="bilinear", align_corners=False)\n        return out[0, 0].cpu().numpy().astype(np.float32)\n',
    'download.py': '"""\nSTEP 1 - download a satellite image for a lat/lon.\n\n    uv run download.py 23.846073, 90.389624 airport_road\n\nWrites:\n    data/images/<name>.tif   georeferenced, used by extract.py\n    data/images/<name>.png   plain picture, just to look at\n\nZoom and size are fixed in config.py (z18, 2048x2048, about 1.1 km across).\n"""\nimport io\nimport math\nimport re\nimport sys\n\nimport numpy as np\nimport rasterio\nimport requests\nfrom PIL import Image\nfrom rasterio.transform import Affine\n\nimport config\nfrom core.geo import ORIGIN, TILE_PX, ground_res, latlon_to_pixel\n\nImage.MAX_IMAGE_PIXELS = None\n\n\ndef _fetch_tile(session, z, x, y):\n    url = config.BASEMAP.format(z=z, x=x, y=y)\n    r = session.get(url, timeout=30)\n    r.raise_for_status()\n    return Image.open(io.BytesIO(r.content)).convert("RGB")\n\n\ndef download(lat, lon, name, zoom=config.ZOOM, size=config.SIZE_PX):\n    # where the image sits in the global pixel grid\n    cx, cy = latlon_to_pixel(lat, lon, zoom)\n    x0, y0 = int(round(cx - size / 2)), int(round(cy - size / 2))\n\n    tx0, ty0 = x0 // TILE_PX, y0 // TILE_PX\n    tx1, ty1 = (x0 + size - 1) // TILE_PX, (y0 + size - 1) // TILE_PX\n    n_tiles = (tx1 - tx0 + 1) * (ty1 - ty0 + 1)\n\n    gsd = ground_res(lat, zoom)\n    print(f"[download] {name}  lat={lat} lon={lon} z={zoom}")\n    print(f"[download] {size}x{size} px  ~{gsd:.2f} m/px  "\n          f"~{size * gsd / 1000:.2f} km across  ({n_tiles} tiles)")\n\n    canvas = Image.new("RGB", ((tx1 - tx0 + 1) * TILE_PX, (ty1 - ty0 + 1) * TILE_PX))\n    session = requests.Session()\n    session.headers["User-Agent"] = config.USER_AGENT\n\n    done = 0\n    for ty in range(ty0, ty1 + 1):\n        for tx in range(tx0, tx1 + 1):\n            tile = _fetch_tile(session, zoom, tx, ty)\n            canvas.paste(tile, ((tx - tx0) * TILE_PX, (ty - ty0) * TILE_PX))\n            done += 1\n            print(f"\\r[download]   tile {done}/{n_tiles}", end="", flush=True)\n    print()\n\n    ox, oy = x0 - tx0 * TILE_PX, y0 - ty0 * TILE_PX\n    img = canvas.crop((ox, oy, ox + size, oy + size))\n\n    # georeference: top-left corner in Web Mercator metres\n    res = 2.0 * ORIGIN / (TILE_PX * 2 ** zoom)\n    west = -ORIGIN + x0 * res\n    north = ORIGIN - y0 * res\n    transform = Affine.translation(west, north) * Affine.scale(res, -res)\n\n    arr = np.asarray(img)\n    tif = config.DATA_IMAGES / f"{name}.tif"\n    with rasterio.open(tif, "w", driver="GTiff", height=size, width=size,\n                       count=3, dtype="uint8", crs="EPSG:3857",\n                       transform=transform, compress="deflate") as dst:\n        dst.write(arr.transpose(2, 0, 1))\n\n    png = config.DATA_IMAGES / f"{name}.png"\n    img.save(png)\n\n    print(f"[download] saved {tif}")\n    print(f"[download] saved {png}")\n    return tif\n\n\nUSAGE = """usage: uv run download.py <lat> <lon> [name]\n\nexamples:\n  uv run download.py 23.846073, 90.389624 airport_road\n  uv run download.py 23.846073 90.389624 airport_road\n  uv run download.py 23.846073,90.389624\n\nPaste coordinates straight from OpenStreetMap or Google - the comma is fine.\nIf you leave out the name, one is made from the coordinates.\n"""\n\n\ndef parse_cli(argv):\n    """Accept \'23.8, 90.3 name\', \'23.8,90.3 name\' or \'23.8 90.3 name\'."""\n    tokens = [t for a in argv for t in re.split(r"[,\\s]+", a) if t]\n    if len(tokens) < 2:\n        sys.exit(USAGE)\n    try:\n        lat, lon = float(tokens[0]), float(tokens[1])\n    except ValueError:\n        sys.exit(USAGE)\n    name = "_".join(tokens[2:]) if len(tokens) > 2 else f"aoi_{lat:.5f}_{lon:.5f}"\n    return lat, lon, re.sub(r"[^0-9A-Za-z._-]", "_", name)\n\n\ndef main():\n    argv = sys.argv[1:]\n    if not argv or argv[0] in ("-h", "--help"):\n        sys.exit(USAGE)\n\n    lat, lon, name = parse_cli(argv)\n    if not (20.0 <= lat <= 27.0 and 88.0 <= lon <= 93.0):\n        print("[warn] coordinates are outside Bangladesh - are lat and lon swapped?",\n              file=sys.stderr)\n\n    download(lat, lon, name)\n\n\nif __name__ == "__main__":\n    main()\n',
    'extract.py': '"""\nSTEP 2 - run a model on a downloaded image and produce roads.\n\n    uv run extract.py airport_road\n    uv run extract.py airport_road baseline\n    uv run extract.py airport_road all\n    uv run extract.py list\n\nWrites:\n    outputs/geojson/<name>_<model>.geojson   <- drag this into iD\n    outputs/images/<name>_<model>.png        <- red overlay, judge by eye\n    outputs/images/<name>_<model>_mask.png\n"""\nimport sys\nimport time\n\nimport numpy as np\nimport rasterio\n\nimport config\nfrom core.geo import raster_center_latlon\nfrom core.registry import discover\nfrom core.tiling import tiled_predict\nfrom core.vectorize import (mask_to_paths, paths_to_geojson, paths_to_mask,\n                            prob_to_mask, save_geojson)\nfrom core.viz import save_mask, save_overlay\n\n\ndef run(name, model_name):\n    tif = config.DATA_IMAGES / f"{name}.tif"\n    if not tif.exists():\n        sys.exit(f"No image at {tif}\\nRun download.py first.")\n\n    with rasterio.open(tif) as src:\n        image = src.read([1, 2, 3]).transpose(1, 2, 0)\n        transform, crs = src.transform, src.crs\n        lat, lon = raster_center_latlon(src)\n\n    models = discover()\n    if model_name not in models:\n        sys.exit(f"Unknown model \'{model_name}\'. Available: {\', \'.join(sorted(models))}")\n\n    print(f"[extract] {name}  model={model_name}")\n    t0 = time.time()\n    model = models[model_name]().load()\n\n    if getattr(model, "outputs", "mask") == "graph":\n        # the model returns road lines directly - no thresholding needed\n        paths = model.predict_graph(image)\n        mask = paths_to_mask(paths, image.shape[:2])\n    else:\n        prob = tiled_predict(model, image, config.PATCH_PX, config.OVERLAP_PX)\n        mask = prob_to_mask(prob, config.THRESHOLD, config.MIN_ROAD_PX)\n        paths = mask_to_paths(mask)\n    fc = paths_to_geojson(paths, transform, crs, lat,\n                          simplify_m=config.SIMPLIFY_M,\n                          props={"model": model_name, "aoi": name})\n\n    stem = f"{name}_{model_name}"\n    gj = save_geojson(fc, config.OUT_GEOJSON / f"{stem}.geojson")\n    ov = save_overlay(image, mask, config.OUT_IMAGES / f"{stem}.png")\n    mk = save_mask(mask, config.OUT_IMAGES / f"{stem}_mask.png")\n\n    total_km = sum(f["properties"]["length_m"] for f in fc["features"]) / 1000.0\n    print(f"[extract] {len(fc[\'features\'])} road lines, {total_km:.2f} km total"\n          f"  ({time.time() - t0:.1f}s)")\n    print(f"[extract] saved {gj}")\n    print(f"[extract] saved {ov}")\n    print(f"[extract] saved {mk}")\n    return gj\n\n\nUSAGE = """usage: uv run extract.py <name> [model]\n\nexamples:\n  uv run extract.py airport_road            # uses the baseline model\n  uv run extract.py airport_road baseline\n  uv run extract.py airport_road all        # every model\n  uv run extract.py list                    # show available models\n\n<name> is the name you gave download.py.\n"""\n\n\ndef main():\n    argv = sys.argv[1:]\n    if not argv or argv[0] in ("-h", "--help"):\n        sys.exit(USAGE)\n\n    models = discover()\n    if argv[0] in ("list", "--list"):\n        for n, cls in sorted(models.items()):\n            print(f"  {n:<12} {cls.description}")\n        return\n\n    name = argv[0]\n    wanted = argv[1] if len(argv) > 1 else "baseline"\n    targets = sorted(models) if wanted == "all" else [wanted]\n    for m in targets:\n        run(name, m)\n\n\nif __name__ == "__main__":\n    main()\n',
}

from pathlib import Path
for relpath, content in FILES.items():
    p = Path(relpath)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
print(f"wrote {len(FILES)} files")


In [ ]:
from pathlib import Path

WEIGHTS_DIR = Path("models/weights")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)   # the actual fix - ensure it exists first

def check_weight(path, min_bytes, label):
    p = Path(path)
    size = p.stat().st_size if p.exists() else 0
    ok = size >= min_bytes
    status = "OK" if ok else "MISSING or TOO SMALL"
    print(f"  {label}: {size / 1e6:.1f} MB  [{status}]")
    if not ok:
        print(f"    ERROR: {label} did not download correctly - re-run this cell, "
              f"or check your network connection.")
    return ok

# D-LinkNet34 (DeepGlobe winner) - zip, needs extracting
dlink_dest = WEIGHTS_DIR / "dlinknet34_deepglobe.th"
if not dlink_dest.exists() or dlink_dest.stat().st_size < 50_000_000:
    print("Downloading D-LinkNet34 weights...")
    !wget -q "https://www.dropbox.com/sh/h62vr320eiy57tt/AAB5Tm43-efmtYzW_GFyUCfma?dl=1" -O /content/dlink.zip
    !mkdir -p /content/dlink_extract
    !unzip -o -q /content/dlink.zip -d /content/dlink_extract
    !cp /content/dlink_extract/log01_dink34.th "{dlink_dest}"
check_weight(dlink_dest, 50_000_000, "D-LinkNet34")

# Plain U-Net (Massachusetts Roads Dataset)
unet_dest = WEIGHTS_DIR / "unet_road_massachusetts.pth"
if not unet_dest.exists() or unet_dest.stat().st_size < 50_000_000:
    print("Downloading U-Net weights...")
    !wget -q "https://huggingface.co/teohyc/Satellite-Road-Segmentation-UNet/resolve/main/best_road_seg_unet.pth" -O "{unet_dest}"
check_weight(unet_dest, 50_000_000, "U-Net")


In [ ]:
# Paste coordinates from OpenStreetMap (right-click a spot -> "Show address") or Google
# Maps - lat and lon together, comma or space separated, e.g. "23.860706, 90.368257".
import re

coords_raw = input("Latitude, Longitude: ").strip()
tokens = [t for t in re.split(r"[,\s]+", coords_raw) if t]
if len(tokens) < 2:
    raise ValueError(f"Need both lat and lon, got: {coords_raw!r}")
lat, lon = float(tokens[0]), float(tokens[1])
name = input("Short name for this area [default: aoi]: ").strip() or "aoi"

import download as dl
dl.download(lat, lon, name)

from IPython.display import Image as IPyImage, display
display(IPyImage(f"data/images/{name}.png", width=500))


In [ ]:
import extract
from core.registry import discover

MODELS = ["baseline", "dlinknet", "unet"]
available = discover()
results = {}

for m in MODELS:
    print(f"--- {m} ({available[m].description}) ---")
    try:
        extract.run(name, m)
        results[m] = {"ok": True, "description": available[m].description}
    except Exception as e:
        print(f"  ERROR: {e}")
        results[m] = {"ok": False, "description": available[m].description, "error": str(e)}


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image as PILImage

ok_models = [m for m in MODELS if results[m]["ok"]]
failed = [m for m in MODELS if not results[m]["ok"]]

if ok_models:
    fig, axes = plt.subplots(2, len(ok_models), figsize=(6 * len(ok_models), 10))
    if len(ok_models) == 1:
        axes = axes.reshape(2, 1)

    for i, m in enumerate(ok_models):
        title = results[m]["description"]   # real model name/description, not the short code
        overlay = PILImage.open(f"outputs/images/{name}_{m}.png")
        mask = PILImage.open(f"outputs/images/{name}_{m}_mask.png")

        axes[0, i].imshow(overlay)
        axes[0, i].set_title(title, fontsize=11)
        axes[0, i].axis("off")

        axes[1, i].imshow(mask, cmap="gray")
        axes[1, i].set_title(f"{title} - mask", fontsize=10)
        axes[1, i].axis("off")

    plt.tight_layout()
    plt.show()

for m in failed:
    print(f"[skipped] {results[m]['description']}: {results[m]['error']}")


In [ ]:
import shutil
from google.colab import files

shutil.make_archive("road_extraction_outputs", "zip", "outputs")
files.download("road_extraction_outputs.zip")
